# Théorie de l'information des réseaux

Ce carnet étudie quatre situations où la théorie classique d'un émetteur vers un récepteur ne suffit plus :

1. **Plusieurs émetteurs vers un récepteur** — le canal à accès multiple (MAC) ;
2. **Un émetteur vers plusieurs récepteurs** — le canal de diffusion dégradé ;
3. **Sources corrélées codées séparément** — le codage de Slepian-Wolf ;
4. **Réseau de liens sans bruit** — le codage réseau et la coupe minimale.

Pour chaque situation, on calcule explicitement les bornes de capacité sur un exemple concret, en s'appuyant uniquement sur `numpy` et la bibliothèque standard.

**Prérequis** : le carnet 01 de la série (entropie, information mutuelle). Les fonctions nécessaires sont redéfinies ci-dessous pour que ce carnet s'exécute seul.

In [1]:
import numpy as np
from itertools import product

def entropie(pmf):
    """Entropie de Shannon en bits ; les termes nuls sont ignorés."""
    pmf = np.asarray(pmf, dtype=float)
    pmf = pmf[pmf > 0]
    return float(-np.sum(pmf * np.log2(pmf)))

def h2(p):
    """Entropie binaire h(p) = -p log2(p) - (1-p) log2(1-p)."""
    if p <= 0.0 or p >= 1.0:
        return 0.0
    return -p * np.log2(p) - (1.0 - p) * np.log2(1.0 - p)

def C(x):
    """Capacité du canal gaussien : C(x) = 1/2 log2(1 + x)."""
    return 0.5 * np.log2(1.0 + x)

# Contrôles
print(f"h2(0.5) = {h2(0.5)}")
print(f"C(15)   = {C(15)}")

h2(0.5) = 1.0
C(15)   = 2.0


## Le canal à accès multiple (MAC)

Deux émetteurs partagent un même canal vers un récepteur unique. La région des débits atteignables $\mathcal{R}(X_1, X_2)$ est définie par :

$$R_1 \le I(X_1;\, Y \mid X_2), \qquad R_2 \le I(X_2;\, Y \mid X_1), \qquad R_1 + R_2 \le I(X_1, X_2;\, Y),$$

qui dessine un **pentagone** dans le plan $(R_1, R_2)$ (LN4 p. 4–10).

**Théorème d'Ahlswede-Liao.** La capacité du MAC est l'enveloppe convexe de l'union de ces pentagones sur toutes les lois factorisées $p(x_1)\,p(x_2)$ (LN4 p. 4–10).

Le **partage du temps** rend la région convexe : si deux points sont atteignables, toute combinaison convexe l'est aussi. Le TDMA (un émetteur à la fois) est un cas particulier du partage du temps (LN4 p. 4–8 / 4–9).

In [2]:
# MAC binaire à effacement : Y = X1 + X2 (ternaire)
# X1, X2 ~ Bernoulli(1/2) indépendants

# --- Loi jointe P(X1, X2, Y) ---
joint = {}
for x1, x2 in product([0, 1], [0, 1]):
    y = x1 + x2
    joint[(x1, x2, y)] = 0.25

# --- Marginales ---
p_x1 = {0: 0.5, 1: 0.5}
p_x2 = {0: 0.5, 1: 0.5}
p_x1x2, p_y, p_x1y, p_x2y = {}, {}, {}, {}
for (x1, x2, y), p in joint.items():
    p_x1x2[(x1, x2)] = p_x1x2.get((x1, x2), 0) + p
    p_y[y]          = p_y.get(y, 0) + p
    p_x1y[(x1, y)]  = p_x1y.get((x1, y), 0) + p
    p_x2y[(x2, y)]  = p_x2y.get((x2, y), 0) + p

print("Loi de Y :", dict(sorted(p_y.items())))

# --- Calcul des trois bornes depuis la loi jointe ---
# I(X1; Y | X2) = Σ p(x1,x2,y) log2[ p(x1,x2,y)·p(x2) / (p(x1,x2)·p(y,x2)) ]
I_X1Y_X2 = 0.0
for (x1, x2, y), p in joint.items():
    ratio = (p * p_x2[x2]) / (p_x1x2[(x1, x2)] * p_x2y[(x2, y)])
    I_X1Y_X2 += p * np.log2(ratio)

# I(X2; Y | X1) = Σ p(x1,x2,y) log2[ p(x1,x2,y)·p(x1) / (p(x1,x2)·p(y,x1)) ]
I_X2Y_X1 = 0.0
for (x1, x2, y), p in joint.items():
    ratio = (p * p_x1[x1]) / (p_x1x2[(x1, x2)] * p_x1y[(x1, y)])
    I_X2Y_X1 += p * np.log2(ratio)

# I(X1,X2; Y) = Σ p(x1,x2,y) log2[ p(x1,x2,y) / (p(x1,x2)·p(y)) ]
I_X1X2_Y = 0.0
for (x1, x2, y), p in joint.items():
    ratio = p / (p_x1x2[(x1, x2)] * p_y[y])
    I_X1X2_Y += p * np.log2(ratio)

print(f"\nI(X1; Y | X2)  = {I_X1Y_X2:.6f}")
print(f"I(X2; Y | X1)  = {I_X2Y_X1:.6f}")
print(f"I(X1,X2; Y)    = {I_X1X2_Y:.6f}")

# --- Coins du pentagone ---
R1_max, R2_max, R_sum = I_X1Y_X2, I_X2Y_X1, I_X1X2_Y
c1 = (R1_max, R_sum - R1_max)
c2 = (R_sum - R2_max, R2_max)
print(f"\nCoin 1 : (R1, R2) = ({c1[0]:.4f}, {c1[1]:.4f})")
print(f"Coin 2 : (R1, R2) = ({c2[0]:.4f}, {c2[1]:.4f})")

Loi de Y : {0: 0.25, 1: 0.5, 2: 0.25}

I(X1; Y | X2)  = 1.000000
I(X2; Y | X1)  = 1.000000
I(X1,X2; Y)    = 1.500000

Coin 1 : (R1, R2) = (1.0000, 0.5000)
Coin 2 : (R1, R2) = (0.5000, 1.0000)


## Lecture du pentagone

La région calculée est $R_1 \le 1$, $R_2 \le 1$, $R_1 + R_2 \le 3/2$, conforme à LN18 p. 18–23.

Le **partage du temps** simple entre les coins extrêmes $(1, 0)$ et $(0, 1)$ ne donne que $R_1 + R_2 \le 1$ : le codage conjoint gagne un demi-bit supplémentaire. Le pentagone dépasse donc strictement le triangle du TDMA.

In [3]:
# MAC multiplicateur : Y = X1 · X2
# X1 ~ Bern(p1), X2 ~ Bern(p2) indépendants
# Y ∈ {0, 1} avec P(Y=1) = p1·p2, donc H(Y) = h2(p1·p2)

grid = np.arange(0.0, 1.05, 0.1)  # {0, 0.1, 0.2, …, 1.0}

best_val = -1.0
best_params = []

for p1 in grid:
    for p2 in grid:
        q = p1 * p2  # P(Y = 1)
        hy = h2(q)
        if hy > best_val + 1e-15:
            best_val = hy
            best_params = [(round(p1, 1), round(p2, 1))]
        elif abs(hy - best_val) < 1e-15:
            best_params.append((round(p1, 1), round(p2, 1)))

print(f"Maximum de H(Y) = I(X1,X2; Y) sur la grille : {best_val:.6f}")
print(f"Atteint pour p1·p2 = 0.5, par exemple :")
for p1, p2 in best_params[:4]:
    print(f"  p1 = {p1}, p2 = {p2}  →  H(Y) = {h2(p1*p2):.6f}")

# Les bornes coïncident (LN4 p. 4–5) : la somme des débits ne dépasse jamais 1.
print(f"\nBorne somme : R1 + R2 ≤ H(Y) ≤ {best_val:.1f} bit")

Maximum de H(Y) = I(X1,X2; Y) sur la grille : 1.000000
Atteint pour p1·p2 = 0.5, par exemple :
  p1 = 0.5, p2 = 1.0  →  H(Y) = 1.000000
  p1 = 1.0, p2 = 0.5  →  H(Y) = 1.000000

Borne somme : R1 + R2 ≤ H(Y) ≤ 1.0 bit


## Le MAC gaussien

Le canal est $Y = g_1 X_1 + g_2 X_2 + Z$ avec $Z \sim \mathcal{N}(0,1)$, et les rapports signal-bruit reçus sont $S_j$ (LN4 p. 4–29).

**Théorème.** La région de capacité est :

$$R_1 \le C(S_1), \qquad R_2 \le C(S_2), \qquad R_1 + R_2 \le C(S_1 + S_2)$$

(LN4 p. 4–30), où $C(x) = \tfrac{1}{2}\log_2(1+x)$.

Les deux coins du pentagone s'atteignent par **décodage successif** : le récepteur décode d'abord un message en traitant l'autre comme du bruit, puis le soustrait avant de décoder le second.

In [4]:
# Région du MAC gaussien — valeurs d'illustration
S1 = 10.0
S2 = 5.0

C_S1  = C(S1)
C_S2  = C(S2)
C_sum = C(S1 + S2)

print(f"C(S1)     = C({S1:.0f})  = {C_S1:.5f}")
print(f"C(S2)     = C({S2:.0f})  = {C_S2:.5f}")
print(f"C(S1+S2)  = C({S1+S2:.0f})  = {C_sum:.5f}")

# Coin 1 : décodage successif X2 puis X1
# R1 = C(S1), R2 = C(S2 / (1 + S1))
R2_coin1 = C(S2 / (1.0 + S1))
coin1 = (C_S1, R2_coin1)

# Coin 2 : décodage successif X1 puis X2
# R1 = C(S1 / (1 + S2)), R2 = C(S2)
R1_coin2 = C(S1 / (1.0 + S2))
coin2 = (R1_coin2, C_S2)

print(f"\nCoin 1 : (R1, R2) = ({coin1[0]:.5f}, {coin1[1]:.5f})  →  somme = {sum(coin1):.5f}")
print(f"Coin 2 : (R1, R2) = ({coin2[0]:.5f}, {coin2[1]:.5f})  →  somme = {sum(coin2):.5f}")
print(f"\nCapacité somme C(S1+S2) = {C_sum:.5f}")

C(S1)     = C(10)  = 1.72972
C(S2)     = C(5)  = 1.29248
C(S1+S2)  = C(15)  = 2.00000

Coin 1 : (R1, R2) = (1.72972, 0.27028)  →  somme = 2.00000
Coin 2 : (R1, R2) = (0.70752, 1.29248)  →  somme = 2.00000

Capacité somme C(S1+S2) = 2.00000


In [5]:
# TDMA, contrôle de puissance, traitement du bruit — MAC gaussien S1=10, S2=5
S1, S2 = 10.0, 5.0

alpha = np.linspace(0.0, 1.0, 1001)  # grille fine sur [0, 1]
bar_alpha = 1.0 - alpha

# 1. TDMA simple : alpha*C(S1) + (1-alpha)*C(S2)
tdma_simple = alpha * C(S1) + bar_alpha * C(S2)
idx_max_simple = np.argmax(tdma_simple)
max_tdma_simple = tdma_simple[idx_max_simple]
alpha_opt_simple = alpha[idx_max_simple]

# 2. TDMA avec contrôle de puissance (LN4 p. 4-33) :
#    alpha*C(S1/alpha) + (1-alpha)*C(S2/(1-alpha))
#    (formule définie sur ]0,1[ ; aux bornures elle dégénère)
mask_int = (alpha > 0) & (alpha < 1)
tdma_pc = np.full_like(alpha, -np.inf)
tdma_pc[mask_int] = (alpha[mask_int] * C(S1 / alpha[mask_int])
                     + bar_alpha[mask_int] * C(S2 / bar_alpha[mask_int]))
idx_max_pc = np.argmax(tdma_pc)
max_tdma_pc = tdma_pc[idx_max_pc]
alpha_opt = alpha[idx_max_pc]

# 3. L'autre signal traité comme du bruit (LN4 p. 4-31) :
#    C(S1/(1+S2)) + C(S2/(1+S1))  — indépendant de alpha
tdma_noise = C(S1 / (1 + S2)) + C(S2 / (1 + S1))

print(f"TDMA simple            : max = {max_tdma_simple:.5f}  à alpha = {alpha_opt_simple:.4f}")
print(f"TDMA + ctrl puissance  : max = {max_tdma_pc:.5f}  à alpha = {alpha_opt:.4f}")
print(f"  (S1/(S1+S2) = {S1/(S1+S2):.4f})")
print(f"Bruit (pas de décodage):       {tdma_noise:.5f}")
print(f"\nCapacité somme C(S1+S2) = {C(S1+S2):.5f}")

TDMA simple            : max = 1.72972  à alpha = 1.0000
TDMA + ctrl puissance  : max = 2.00000  à alpha = 0.6670
  (S1/(S1+S2) = 0.6667)
Bruit (pas de décodage):       0.97780

Capacité somme C(S1+S2) = 2.00000


## Le canal de diffusion dégradé

Un émetteur unique envoie vers deux récepteurs dont les canaux ont des qualités différentes. La technique clé est le **codage par superposition** (LN5 p. 5–13) : on empile deux « nuages » de codage, l'un pour le récepteur faible, l'autre pour le récepteur fort.

**Capacité du canal dégradé.** Si $U$ désigne le nuage externe (message du récepteur faible) et $X$ le signal complet :

$$R_1 \le I(X;\, Y_1 \mid U), \qquad R_2 \le I(U;\, Y_2)$$

(LN5 p. 5–21).

**Cas gaussien.** Le récepteur 1 est le plus fort : $N_1 \le N_2$ (LN5 p. 5–29). Avec une allocation de puissance $\alpha P$ pour le message fort et $\bar{\alpha} P$ pour le message faible :

$$R_1 \le C(\alpha S_1), \qquad R_2 \le C\!\left(\frac{\bar{\alpha}\, S_2}{\alpha S_2 + 1}\right)$$

(LN5 p. 5–31), où $S_j = P/N_j$.

Le nuage $U$ (puissance $\bar{\alpha}P$) porte le message du récepteur faible. Le récepteur fort le décode d'abord, puis le **soustrait** avant de décoder son propre message.

In [6]:
# Diffusion gaussienne vs partage du temps
# P = 10, N1 = 1, N2 = 4 (valeurs d'illustration)
P, N1, N2 = 10.0, 1.0, 4.0
S1_bc = P / N1   # RSB récepteur fort  = 10
S2_bc = P / N2   # RSB récepteur faible = 2.5

C_S1_bc = C(S1_bc)  # C(10)
C_S2_bc = C(S2_bc)  # C(2.5)

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]

print(f"P={P:.0f}, N1={N1:.0f}, N2={N2:.0f}  →  S1={S1_bc:.1f}, S2={S2_bc:.1f}")
print(f"C(S1)={C_S1_bc:.5f}, C(S2)={C_S2_bc:.5f}\n")

print(f"{'alpha':>5} | {'R1 sup':>8} | {'R2 sup':>8} | {'R2 TDMA':>8} | {'gain':>7}")
print("-" * 50)

strictly_dominates_interior = True
for a in alphas:
    r1 = C(a * S1_bc)
    r2_sup = C((1 - a) * S2_bc / (a * S2_bc + 1.0))
    # Partage du temps : segment entre (C(S1),0) et (0,C(S2))
    # Au même R1 : R2_tdma = C(S2) * (1 - R1/C(S1))
    r2_tdma = C_S2_bc * (1.0 - r1 / C_S1_bc) if C_S1_bc > 0 else 0.0
    gain = r2_sup - r2_tdma
    print(f"{a:5.2f} | {r1:8.5f} | {r2_sup:8.5f} | {r2_tdma:8.5f} | {gain:+7.5f}")
    if 0 < a < 1 and gain <= 1e-12:
        strictly_dominates_interior = False

# Vérification sur grille fine : domination stricte sur les 999 points échantillonnés de ]0,1[
alpha_fine = np.linspace(0.001, 0.999, 999)
r1_fine = C(alpha_fine * S1_bc)
r2_sup_fine = C((1 - alpha_fine) * S2_bc / (alpha_fine * S2_bc + 1.0))
r2_tdma_fine = C_S2_bc * (1.0 - r1_fine / C_S1_bc)
min_gain_fine = float(np.min(r2_sup_fine - r2_tdma_fine))

print(f"\nGain minimal sur les 999 points échantillonnés dans ]0,1[ : {min_gain_fine:.6f}")
print(f"Superposition domine strictement aux 3 points intérieurs (0 < alpha < 1) : {strictly_dominates_interior}")
print(f"Égalité aux extrémités (alpha=0 et alpha=1) : gain = 0 (visible dans le tableau).")

P=10, N1=1, N2=4  →  S1=10.0, S2=2.5
C(S1)=1.72972, C(S2)=0.90368

alpha |   R1 sup |   R2 sup |  R2 TDMA |    gain
--------------------------------------------------
 0.00 |  0.00000 |  0.90368 |  0.90368 | +0.00000
 0.25 |  0.90368 |  0.55346 |  0.43156 | +0.12190
 0.50 |  1.29248 |  0.31871 |  0.22843 | +0.09028
 0.75 |  1.54373 |  0.14190 |  0.09717 | +0.04473
 1.00 |  1.72972 |  0.00000 |  0.00000 | +0.00000

Gain minimal sur les 999 points échantillonnés dans ]0,1[ : 0.000173
Superposition domine strictement aux 3 points intérieurs (0 < alpha < 1) : True
Égalité aux extrémités (alpha=0 et alpha=1) : gain = 0 (visible dans le tableau).


## Slepian-Wolf

Deux sources corrélées $X_1$ et $X_2$ sont codées **séparément** (chaque encodeur ne voit que sa propre source), mais décodées **ensemble** par un décodeur conjoint. La région des débits atteignables est :

$$R_1 \ge H(X_1 \mid X_2), \qquad R_2 \ge H(X_2 \mid X_1), \qquad R_1 + R_2 \ge H(X_1, X_2)$$

(LN11 p. 11–9).

Le résultat surprenant : le décodeur conjoint atteint exactement le débit du codage conjoint ($H(X_1, X_2)$), sans que les encodeurs échangent d'information. L'économie provient de la corrélation, exploitée uniquement côté décodeur.

In [7]:
# Source binaire doublement symétrique (DSBS) avec p = 0.11
p = 0.11

# Loi jointe : P(X1=i, X2=j) = (1-p)/2 si i=j, p/2 si i≠j
joint_dsbs = {
    (0, 0): (1 - p) / 2,
    (0, 1): p / 2,
    (1, 0): p / 2,
    (1, 1): (1 - p) / 2,
}

print("Loi jointe DSBS(p=0.11) :")
for (x1, x2), prob in sorted(joint_dsbs.items()):
    print(f"  P(X1={x1}, X2={x2}) = {prob:.4f}")

# Marginales et conditionnelles
p_x1_0 = joint_dsbs[(0,0)] + joint_dsbs[(0,1)]
p_x1_1 = joint_dsbs[(1,0)] + joint_dsbs[(1,1)]
H_X1 = entropie([p_x1_0, p_x1_1])

# H(X2|X1) = H(X1,X2) - H(X1)
H_joint = entropie(list(joint_dsbs.values()))
H_X2_given_X1 = H_joint - H_X1

print(f"\nH(X1)        = {H_X1:.5f}")
print(f"H(X2|X1)     = {H_X2_given_X1:.5f}   (= h2({p}))")
print(f"H(X1,X2)     = {H_joint:.5f}")

# Pour 100 bits de chaque source (estimation asymptotique du débit)
n_bits = 100
code_indep = n_bits * H_X1 + n_bits * H_X1  # codage indépendant (symétrique)
code_sw = n_bits * H_joint                    # Slepian-Wolf

print(f"\nPour {n_bits} bits de chaque source (estimation asymptotique) :")
print(f"  Codage indépendant : {code_indep:.2f} bits")
print(f"  Slepian-Wolf       : {code_sw:.2f} bits  (≈ 150)")

# Deux coins de la région
coin_a = (H_X2_given_X1, H_X1)   # (H(X2|X1), H(X1))
coin_b = (H_X1, H_X2_given_X1)   # (H(X1), H(X2|X1))
print(f"\nCoins de la région : ({coin_a[0]:.5f}, {coin_a[1]:.5f}) et ({coin_b[0]:.5f}, {coin_b[1]:.5f})")

Loi jointe DSBS(p=0.11) :
  P(X1=0, X2=0) = 0.4450
  P(X1=0, X2=1) = 0.0550
  P(X1=1, X2=0) = 0.0550
  P(X1=1, X2=1) = 0.4450

H(X1)        = 1.00000
H(X2|X1)     = 0.49992   (= h2(0.11))
H(X1,X2)     = 1.49992

Pour 100 bits de chaque source (estimation asymptotique) :
  Codage indépendant : 200.00 bits
  Slepian-Wolf       : 149.99 bits  (≈ 150)

Coins de la région : (0.49992, 1.00000) et (1.00000, 0.49992)


## Codage réseau

Dans un réseau de liens sans bruit, la capacité d'acheminement vers un destinataire unique est donnée par la **coupe minimale** (théorème max-flow / min-cut, LN16 p. 16–5).

En **multidiffusion** (un émetteur vers plusieurs destinataires), la borne de coupe s'applique à **chaque** destinataire individuellement (LN16 p. 16–11). Cependant, elle n'est pas toujours atteinte par un simple routage : les flux vers les différents destinataires peuvent se concurrencer sur les mêmes liens.

Le **théorème du codage réseau** (LN16 p. 16–15) établit qu'un codage linéaire local (combinaisons linéaires des paquets reçus) permet d'atteindre la borne de coupe pour **tous** les destinataires simultanément, sans erreur. C'est le codage réseau.

In [8]:
# Réseau papillon (LN16 p. 16-13) : coupe minimale
# Nœuds : 1 (source), 2, 3, 4, 5, 6 (dest), 7 (dest)
# Arêtes (capacité 1 chacune) :
edges = [(1,2), (1,3), (2,4), (3,4), (4,5), (2,6), (3,7), (5,6), (5,7)]
nodes = set(range(1, 8))

def cut_capacity(S, j):
    """Nombre d'arêtes allant de S à son complément (S contient 1, pas j)."""
    return sum(1 for (u, v) in edges if u in S and v not in S)

for j in [6, 7]:
    # Énumérer toutes les coupes : S contient 1, exclut j
    others = nodes - {1, j}  # {2,3,4,5} \ {j}
    min_cut = float('inf')
    min_S = None
    count = 0
    for r in range(len(others) + 1):
        for combo in __import__('itertools').combinations(others, r):
            S = {1} | set(combo)
            if j in S:
                continue
            cap = cut_capacity(S, j)
            count += 1
            if cap < min_cut:
                min_cut = cap
                min_S = S
    print(f"Destinataire j={j} : {count} coupes énumérées, coupe minimale = {min_cut}")
    print(f"  Exemple de coupe min : S = {sorted(min_S)}")
    print()

Destinataire j=6 : 32 coupes énumérées, coupe minimale = 2
  Exemple de coupe min : S = [1]

Destinataire j=7 : 32 coupes énumérées, coupe minimale = 2
  Exemple de coupe min : S = [1]



In [9]:
# Le papillon : routage contre XOR sur le goulot 4→5
import numpy as np

np.random.seed(42)
k = 100  # longueur des blocs en bits
M1 = np.random.randint(0, 2, k)  # message destiné au nœud 6
M2 = np.random.randint(0, 2, k)  # message destiné au nœud 7

# Topologie du papillon :
#   1→2 transporte M1,  1→3 transporte M2
#   2→4 transporte M1,  2→6 transporte M1
#   3→4 transporte M2,  3→7 transporte M2
#   4→5 transporte [stratégie],  5→6 et 5→7 relaient ce qu'a reçu 5
#
# Réception :
#   Nœud 6 : reçoit M1 (via 2→6) + signal_goulot (via 5→6)
#   Nœud 7 : reçoit M2 (via 3→7) + signal_goulot (via 5→7)

def simuler(strategie):
    """Simule la réception en 6 et 7.
    Renvoie (nb_dest_corrects, nb_erreurs_bits, nb_messages_absents)."""
    if strategie == 'M1':
        goulot = M1.copy()
    elif strategie == 'M2':
        goulot = M2.copy()
    else:  # 'XOR'
        goulot = M1 ^ M2

    # Nœud 6 : a M1 (direct via 2→6) + goulot (via 5→6)
    rec6_M1 = M1.copy()  # toujours correct (chemin direct 2→6)
    if strategie == 'M1':
        rec6_M2 = None   # goulot porte M1, impossible de récupérer M2
    elif strategie == 'M2':
        rec6_M2 = goulot.copy()  # goulot porte M2 → correct
    else:
        rec6_M2 = M1 ^ goulot    # M1 ^ (M1^M2) = M2

    # Nœud 7 : a M2 (direct via 3→7) + goulot (via 5→7)
    rec7_M2 = M2.copy()  # toujours correct (chemin direct 3→7)
    if strategie == 'M1':
        rec7_M1 = goulot.copy()  # goulot porte M1 → correct
    elif strategie == 'M2':
        rec7_M1 = None           # goulot porte M2, impossible de récupérer M1
    else:
        rec7_M1 = M2 ^ goulot    # M2 ^ (M1^M2) = M1

    # Comptage : erreurs de comparaison vs messages absents (effacement)
    def stats(rec, original):
        """Renvoie (nb_erreurs_bits, nb_absent) : 0/1 si absent, sinon compte les bits faux."""
        if rec is None:
            return 0, 1  # message absent (effacement), aucune comparaison possible
        return int(np.sum(rec != original)), 0

    err6, abs6 = stats(rec6_M2, M2)  # M2 reconstruit vs M2 original
    err7, abs7 = stats(rec7_M1, M1)  # M1 reconstruit vs M1 original

    # Un destinataire « réussit » s'il a les DEUX messages sans erreur
    ok6 = (rec6_M2 is not None) and (np.all(rec6_M1 == M1) and np.all(rec6_M2 == M2))
    ok7 = (rec7_M1 is not None) and (np.all(rec7_M2 == M2) and np.all(rec7_M1 == M1))

    nb_ok = int(ok6) + int(ok7)
    return nb_ok, err6 + err7, abs6 + abs7

print(f"Blocs de {k} bits, graine 42\n")
print(f"{'Stratégie 4→5':<16} | {'Dest. corrects':<15} | {'Err. bits':<10} | {'Msg. absents'}")
print("-" * 55)
for strat in ['M1', 'M2', 'XOR']:
    nb_ok, nb_err, nb_abs = simuler(strat)
    label = f"transmettre {strat.replace('XOR','M1⊕M2')}"
    print(f"{label:<16} | {nb_ok}/2              | {nb_err:<10} | {nb_abs}")

print(f"\nDébit : routage → 1 bit/symbole ; XOR → 2 bits/symbole (LN16 p. 16-14)")

Blocs de 100 bits, graine 42

Stratégie 4→5    | Dest. corrects  | Err. bits  | Msg. absents
-------------------------------------------------------
transmettre M1   | 1/2              | 0          | 1
transmettre M2   | 1/2              | 0          | 1
transmettre M1⊕M2 | 2/2              | 0          | 0

Débit : routage → 1 bit/symbole ; XOR → 2 bits/symbole (LN16 p. 16-14)


## Vérifiez votre compréhension

**Exercice 1 — MAC binaire à effacement.** La région du pentagone est $R_1 \le 1$, $R_2 \le 1$, $R_1 + R_2 \le 3/2$. Le point $(0{,}75\;;\;0{,}75)$ est-il atteignable ? Et le point $(1\;;\;0{,}6)$ ? Justifier en vérifiant les trois inégalités.

**Exercice 2 — MAC gaussien symétrique.** Soit $S_1 = S_2 = S$. Quel choix de $\alpha$ dans la formule de TDMA avec contrôle de puissance, $\alpha\, C(S_1/\alpha) + \bar{\alpha}\, C(S_2/\bar{\alpha})$, atteint la capacité somme $C(2S)$ ? Pourquoi ce choix est-il naturel ?


In [10]:
def exercice_1(r1, r2):
    """MAC à effacement : le point (r1, r2) est-il dans le pentagone ?
    À compléter."""
    print(f"exercice_1({r1}, {r2}) : à compléter")
    return None

def exercice_2(S):
    """MAC gaussien symétrique : alpha optimal du TDMA avec contrôle de puissance.
    À compléter."""
    print(f"exercice_2(S={S}) : à compléter")
    return None

# Appels sur les valeurs de l'énoncé
print("--- Exercice 1 ---")
r = exercice_1(0.75, 0.75)
print(f"retour : {r}\n")
r = exercice_1(1.0, 0.6)
print(f"retour : {r}\n")

print("--- Exercice 2 ---")
r = exercice_2(5.0)
print(f"retour : {r}")

--- Exercice 1 ---


exercice_1(0.75, 0.75) : à compléter
retour : None

exercice_1(1.0, 0.6) : à compléter
retour : None

--- Exercice 2 ---
exercice_2(S=5.0) : à compléter
retour : None


## Synthèse

Quatre idées clés de ce carnet :

1. **Le pentagone du MAC.** Plusieurs émetteurs vers un récepteur : la région des débits est un polygone borné par des informations mutuelles conditionnelles. Sur l'exemple du MAC binaire à effacement, le codage conjoint dépasse strictement le triangle du TDMA.

2. **La superposition en diffusion.** Un émetteur vers plusieurs récepteurs de qualité différente : empiler les messages (nuages de codage) et laisser le récepteur fort soustraire le message du faible avant de décoder le sien. Sur l'exemple gaussien, la superposition domine strictement le partage du temps pour tout $\alpha \in\,]0,1[$.

3. **Le décodage conjoint de Slepian-Wolf.** Deux sources corrélées codées séparément atteignent, grâce au décodeur conjoint, exactement le débit du codage conjoint — sans communication entre encodeurs.

4. **La coupe minimale et le codage réseau.** Dans un réseau sans bruit, la borne de max-flow/min-cut s'applique à chaque destinataire en multidiffusion. Sur le réseau papillon, le simple routage n'atteint pas la borne pour tous les destinataires simultanément ; un codage linéaire local (XOR généralisé) y parvient.

> Carnet issu de l'EPIC #27077.


## À retenir

- **Le pentagone du MAC** : la région des débits est bornée par des informations
  mutuelles conditionnelles — sur le MAC binaire à effacement, `R1 ≤ 1`, `R2 ≤ 1`,
  `R1 + R2 ≤ 3/2` — et le codage conjoint y dépasse strictement le triangle du TDMA.
- **La superposition en diffusion** : empiler les messages (nuages de codage) domine
  strictement le partage du temps — mesuré sur grille : gain strict aux 999 points
  intérieurs (minimum `0.000173`), égalité seule aux extrémités.
- **Slepian-Wolf** : les sources corrélées se codent séparément aux taux conditionnels —
  sur l'exemple DSBS `p = 0.11`, `149,99` bits pour 2×100 bits contre `200` en
  codage indépendant.
- **Le codage réseau** : sur le papillon, le routage livre `1` message sur `2`
  destinataires, l'XOR au goulot livre les `2/2`.


## Pour aller plus loin

**Source unique.** A. El Gamal et Y.-H. Kim, *Lecture Notes on Network Information
Theory*, arXiv:1001.3404v4 (juin 2010), copie archivée au gisement, rayon
`Bibliographie IA/InformationTheory/`.

Le carnet 09 de la série aborde la sécurité au sens de l'information (secret-key
agreement, wiretap channel).
